# Prompt Complexity Classifier & RouterBench (English)

This notebook demonstrates:
1. **RouterBench English Benchmark**: 392,832 rows spanning GSM8K (math), MBPP (Python coding), Hellaswag (reasoning), MMLU, and ARC Challenge.
2. **Prompt Classifier**: Scoring prompt complexity across 14 parameters and routing to **Nano**, **Super**, or **Ultra** tiers.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is in sys.path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

## 1. Load RouterBench English Dataset
We load `data/routerbench/routerbench_0shot_english.parquet` (392,832 rows of pure English benchmark prompts).

In [ ]:
# Load English benchmark dataset
df_en = pd.read_parquet("data/routerbench/routerbench_0shot_english.parquet")
print(f"Loaded English RouterBench: {len(df_en):,} rows x {len(df_en.columns)} columns")

# Preview top English benchmarks: GSM8K (math), MBPP (coding), and Hellaswag
sample_benchmarks = ["grade-school-math", "mbpp", "arc-challenge", "hellaswag"]
preview_rows = df_en[df_en["dataset"].isin(sample_benchmarks)].groupby("dataset").head(1)
preview_rows[["dataset", "prompt", "model", "correct", "cost"]]

## 2. Test Prompt Classifier on Real English Prompts
Score prompts across 14 parameters and route to **Nano**, **Super**, or **Ultra** models.

In [ ]:
from tools.prompt_classifier import classify_prompt

sample_prompts = [
    "What is a put option? brief answer",
    "Write a Python function to find the intersection of two sorted arrays in O(N) time.",
    "Optimize a long-only systematic strategy under a 10% max drawdown limit. Derive sizing equations and write the backtest."
]

for prompt in sample_prompts:
    res = classify_prompt(prompt)
    print("=" * 75)
    print(f"PROMPT:            {prompt}")
    print(f"Route Tier:        {res.route_tier.upper()}")
    print(f"Recommended Model: {res.recommended_model}")
    print(f"Weighted Score:    {res.weighted_score:.3f}")
    print(f"Task Type:         {res.scores.predicted_task_type}")

## 3. Interactive Custom Prompt & 14-Parameter Breakdown
Input your own prompt to see the complete rubric breakdown.

In [ ]:
my_prompt = "Design an automated order routing microservice in Python with asyncio, redis pub/sub, and circuit breaker retry logic."

res = classify_prompt(my_prompt)
print(f"Result: {res.route_tier.upper()} -> {res.recommended_model} (Weighted Score: {res.weighted_score:.3f})")

# Display parameter score table
scores_dict = res.to_dict()["scores"]
scores_df = pd.DataFrame(list(scores_dict.items()), columns=["Rubric Parameter", "Score"])
scores_df